# CreditIQ - Colab Master Run (one notebook, end to end)

**What this does:** mounts Drive -> gets the Home Credit data -> runs every pipeline stage -> checks the results for sanity -> runs the test-suite -> generates the documentation from the *executed* artifacts -> launches the API + dashboard -> exports a portable bundle.

**How to use:** `Runtime -> Run all` (CPU runtime is enough, no GPU needed). Every stage checkpoints to Google Drive, so if Colab disconnects just run again: finished stages are skipped.

**Before the first run:** (1) put `CreditIQ.zip` in `MyDrive/` (or upload it when asked), (2) add Colab Secrets `KAGGLE_USERNAME` + `KAGGLE_KEY` and accept the competition rules on Kaggle - or upload the 8 CSVs to `MyDrive/CreditIQ/data/raw/`.

Full guide: `docs/COLAB_RUNBOOK.md`.

## 0. Settings

In [ ]:
PROFILE = "fast"          # "fast": baseline-first run (do this first) | "max": full dev set + Optuna-tuned LightGBM (run after fast)
FORCE = False             # True = ignore checkpoints and recompute every stage
LAUNCH_APPS = True        # start FastAPI + Streamlit at the end (Colab only)
SYNTHETIC_SMOKE = False   # ONLY for testing the notebook mechanics locally with fake data. Never on Colab, never report its numbers.
assert PROFILE in ("fast", "max")

## 1. Environment: Drive, project, dependencies

In [ ]:
import os, sys, subprocess, shutil, time, json, zipfile
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    assert not SYNTHETIC_SMOKE, "SYNTHETIC_SMOKE must stay False on Colab (it would overwrite raw data)."
    DRIVE = Path("/content/drive/MyDrive"); ROOT = DRIVE / "CreditIQ"
    if not (ROOT / "src").exists():
        cands = [DRIVE / "CreditIQ.zip", *Path("/content").glob("CreditIQ*.zip")]
        zp = next((z for z in cands if z.exists()), None)
        if zp is None:
            from google.colab import files
            print("CreditIQ.zip not found - upload it now:"); up = files.upload(); zp = Path("/content") / next(iter(up))
        with zipfile.ZipFile(zp) as z: z.extractall(DRIVE)       # zip root folder is CreditIQ/
        print("Project unzipped to", ROOT)
else:
    ROOT = Path(os.environ.get("CREDITIQ_ROOT", Path.cwd().parent))

assert (ROOT / "src").exists(), f"Project not found at {ROOT}"
os.environ["CREDITIQ_ROOT"] = str(ROOT)
os.environ["CREDITIQ_CONFIG"] = "config_max.yaml" if PROFILE == "max" else "config.yaml"
sys.path.insert(0, str(ROOT / "src"))

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], capture_output=True, text=True)
print("pip install:", "ok" if r.returncode == 0 else r.stderr[-600:])

from creditiq import pipeline
from creditiq.config import get_paths, load_config, read_state, read_json, is_done
paths = get_paths(); cfg = load_config(paths)
print(f"Project root: {ROOT}\nProfile: {PROFILE} -> models={cfg['models']}, cv.sample_n={cfg['cv'].get('sample_n')}, optuna_trials={cfg['tuning']['optuna_trials']}")

In [ ]:
import psutil
print(f"RAM available: {psutil.virtual_memory().available/1e9:.1f} GB | Disk free: {shutil.disk_usage('/').free/1e9:.0f} GB | CPUs: {os.cpu_count()}")
if IN_COLAB and psutil.virtual_memory().available < 6e9:
    print("WARNING: < 6 GB RAM free. Restart the runtime (Runtime -> Restart session) and run again.")

In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown
pd.set_option("display.max_columns", 50, "display.width", 200)
TRAINED_NOW = False

def stage(name, **kw):
    """Run a stage unless it is already checkpointed (or FORCE). Downstream stages re-run if training just re-ran."""
    redo = FORCE or (TRAINED_NOW and name in ("analytics", "report"))
    if is_done(paths, name) and not redo and name != "export":
        print(f"[skip] {name} (already done)"); return
    pipeline.run(name, force=redo and name in ("ingest", "aggregate"), **kw)

## 2. Data: acquire, validate, audit, aggregate, features
Kaggle API download needs the secrets + accepted rules; otherwise the stage prints manual-upload instructions and stops (upload the 8 CSVs to `data/raw/` and re-run this cell). The aggregation step runs DuckDB SQL over ~40M rows; it streams from Parquet so RAM stays low.

In [ ]:
t0 = time.time()
if SYNTHETIC_SMOKE:
    stage("ingest", synthetic=True)
else:
    stage("ingest")
for s in ["audit", "aggregate", "features"]:
    stage(s)
print(f"data stages done in {time.time()-t0:.0f}s")

In [ ]:
display(pd.read_csv(paths.reports / "file_validation.csv"))
display(pd.read_csv(paths.reports / "dataset_audit.csv"))
display(pd.read_csv(paths.reports / "relationship_checks.csv"))
print("Train/test consistency:", read_json(paths.metrics / "train_test_consistency.json"))
la = pd.read_csv(paths.reports / "leakage_audit.csv"); print(f"Leakage audit: {len(la)} rows"); display(la.head(20))

## 3. EDA and statistical tests

In [ ]:
stage("eda"); stage("stats")
display(pd.read_csv(paths.reports / "statistical_tests.csv").head(15))
display(pd.read_csv(paths.reports / "top_correlations.csv").head(15))
print("Core questions:", json.dumps(read_json(paths.reports / "core_questions.json"), indent=1)[:1500])
for f in sorted(paths.figures.glob("*.png"))[:6]: display(Image(str(f)))

## 4. Modelling, calibration, evaluation
`fast` = the config in `configs/config.yaml` (CV on a 120k stratified sample, no tuning). `max` = `configs/config_max.yaml` (whole development set, Optuna-tuned LightGBM). Switching profile automatically forces a retrain and keeps a copy of the previous metrics in `artifacts/runs/`.

In [ ]:
marker = paths.logs / "train_profile.txt"
prev = marker.read_text().strip() if marker.exists() else None
need_train = FORCE or not is_done(paths, "train") or (prev is not None and prev != PROFILE)
if need_train:
    if prev and prev != PROFILE and (paths.metrics / "metrics.json").exists():
        bk = paths.root / "artifacts" / "runs" / prev
        if not bk.exists(): shutil.copytree(paths.metrics, bk); print("Previous run metrics backed up to", bk)
    t0 = time.time(); pipeline.run("train", force=True); TRAINED_NOW = True
    marker.write_text(PROFILE); print(f"training done in {(time.time()-t0)/60:.1f} min")
else:
    print("[skip] train (already done with profile", prev, ")")

In [ ]:
m = read_json(paths.metrics / "metrics.json"); tc = m["test_calibrated"]
syn = read_state(paths)["completed"].get("ingest", {}).get("synthetic", False)
print(f"Selected model: {m['selected_model']} | calibration: {m['calibration_method']} | threshold: {m['decision_threshold']:.4f} | splits: {m['split_sizes']}")
display(pd.Series({k: round(tc[k], 4) for k in ["roc_auc","pr_auc","ks","brier","log_loss","precision","recall","f1","specificity"] if k in tc}, name="holdout (calibrated)").to_frame())
auc = tc["roc_auc"]
if syn:      print("SYNTHETIC DATA - these numbers are meaningless.")
elif auc > 0.85: print(f"SUSPICIOUS: holdout AUC {auc:.3f} is far above what this dataset normally gives -> check leakage_audit.csv before trusting anything.")
elif auc < 0.70: print(f"LOW: holdout AUC {auc:.3f} is below the usual range -> check ingestion/aggregation (row counts, missing tables).")
else:        print(f"OK: holdout AUC {auc:.3f} is in the plausible range for Home Credit.")
display(pd.read_csv(paths.metrics / "cv_comparison.csv")); display(pd.read_csv(paths.metrics / "holdout_comparison.csv"))
if m.get("tuned_params"): print("Optuna best params:", m["tuned_params"])
imb = paths.metrics / "imbalance_experiment.csv"
if imb.exists(): display(pd.read_csv(imb))

In [ ]:
for f in ["roc", "pr", "calibration", "confusion"]: display(Image(str(paths.figures / f"{f}.png")))
display(pd.read_csv(paths.metrics / "risk_deciles_test.csv")); display(pd.read_csv(paths.metrics / "threshold_analysis.csv").head(12))

## 5. Explainability, risk bands, ECL, portfolio, fairness, drift

In [ ]:
stage("analytics")
display(pd.read_csv(paths.metrics / "shap_global_importance.csv").head(15)); display(Image(str(paths.figures / "shap_summary.png")))
display(pd.read_csv(paths.reports / "portfolio_by_risk_band.csv"))
print("ECL (illustrative):", json.dumps(read_json(paths.metrics / "ecl_summary.json"), indent=1))
display(pd.read_csv(paths.reports / "fairness_disparity.csv"))
dr = pd.read_csv(paths.metrics / "drift_report.csv"); print("Drift alerts:", int((dr.alert == "ALERT").sum())); display(dr.head(10))

## 6. Automated tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], cwd=ROOT, capture_output=True, text=True, env={**os.environ})
print("\n".join(r.stdout.strip().splitlines()[-6:])); print("pytest exit code:", r.returncode)

## 7. Documentation (generated from executed artifacts) and export

In [ ]:
stage("report"); pipeline.run("export")
docs = sorted(paths.docs.glob("*.md")); print(f"{len(docs)} docs:", [d.name for d in docs])
display(Markdown((paths.docs / "MODEL_CARD.md").read_text()[:4000]))
display(Markdown((paths.docs / "RESUME_BULLETS.md").read_text()))

In [ ]:
st = pd.DataFrame(read_state(paths)["completed"]).T
display(st[[c for c in ["at", "seconds"] if c in st.columns]])
zips = sorted(paths.exports.glob("creditiq_export_*.zip")); print("Export bundle:", zips[-1] if zips else None)
if IN_COLAB and zips:
    from google.colab import files; files.download(str(zips[-1]))

## 8. Launch API (8000) and dashboard (8501)
Colab proxies the ports privately to your session. Not permanent hosting - take screenshots for the README while it is running.

In [ ]:
if LAUNCH_APPS and IN_COLAB:
    import requests
    api = subprocess.Popen([sys.executable, "-m", "uvicorn", "api.main:app", "--port", "8000"], cwd=ROOT, env={**os.environ})
    dash = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "dashboard/app.py", "--server.port", "8501", "--server.headless", "true",
                             "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"], cwd=ROOT, env={**os.environ})
    time.sleep(20); print("API health:", requests.get("http://localhost:8000/health", timeout=10).json())
    from google.colab import output
    output.serve_kernel_port_as_window(8501)
else:
    print("Skipped (LAUNCH_APPS=False or not on Colab).")

In [ ]:
# Run when you are done with the dashboard:
# api.terminate(); dash.terminate()